In [ ]:
!pip install nolds librosa pandas numpy scipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.7/225.7 kB 5.6 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import librosa
import nolds
from scipy.spatial.distance import pdist, squareform
import glob
import os

def p(f):
    y, sr = librosa.load(f, sr=None)
    f0, _, _ = librosa.pyin(y, fmin=65, fmax=600)
    f0 = f0[~np.isnan(f0)]
    f0 = f0[f0 > 0]
    l = np.log(f0)
    d = np.diff(l)
    h, b = np.histogram(d, bins=30, density=True)
    pr = h * np.diff(b)
    pr = pr[pr > 0]
    return -np.sum(pr * np.log2(pr))

def d_f(a):
    return nolds.dfa(a)

def r(a, t=1, dm=3, ep=0.1):
    n = len(a)
    eb = np.array([a[i:n - (dm - 1) * t + i] for i in range(dm)]).T
    ds = squareform(pdist(eb, metric='euclidean'))
    rm = (ds < ep).astype(int)
    rt = []
    for i in range(len(rm)):
        rtn = np.where(rm[i, i+1:] == 1)[0]
        if len(rtn) > 0:
            rt.extend(np.diff(rtn))
    h, b = np.histogram(rt, bins=50, density=True)
    pr = h * np.diff(b)
    pr = pr[pr > 0]
    hm = np.log2(len(pr)) if len(pr) > 1 else 1
    return -np.sum(pr * np.log2(pr)) / hm

if __name__ == '__main__':
    d_in = input("Enter audio folder path: ")
    fls = glob.glob(d_in + "/*.wav")
    res = []
    for fl in fls:
        bn = os.path.basename(fl)
        y, sr = librosa.load(fl, sr=None)
        m = len(y) // 2
        ys = y[max(0, m - sr//2) : min(len(y), m + sr//2)]
        ep = 0.1 * np.std(ys)
        p_val = p(fl)
        d_val = d_f(y)
        r_val = r(ys, 1, 3, ep)
        res.append({"Filename": bn, "RPDE": r_val, "DFA": d_val, "PPE": p_val})
    df = pd.DataFrame(res)
    df.to_csv("out.csv", index=False)
    print("Done")